# Clase 4 — SARSA, Q-Learning y exploración $\varepsilon$-greedy (versión R)

**Curso:** Aprendizaje por Refuerzo: Fundamentos y Aplicaciones
**Institución:** Universidad Austral — Facultad de Ingeniería (Posgrados)
**Docente:** Dr. Darío Ezequiel Díaz
**Fecha:** 26 de mayo de 2026

---

## Resumen del cuaderno

Esta es la contraparte en **R** del cuaderno Python `Clase04_SARSA_QLearning.ipynb`. Reproduce idiomáticamente sus ocho bloques y traslada al código los algoritmos de **control libre de modelo**. Damos el salto desde el problema de *predicción* (estimar $V^\pi$ para una política dada, abordado en la Clase 3) hacia el problema de *control*:

> **¿Cómo aprender una política óptima a partir de la experiencia, sin modelo de la dinámica, equilibrando exploración y explotación?**

Los dos protagonistas son **SARSA** (control *on-policy*) y **Q-Learning** (control *off-policy*), enfrentados en el entorno *Cliff Walking* de Sutton & Barto (ejemplo 6.6). La implementación emplea:

- **`R6`** para el entorno orientado a objetos.
- **`ggplot2`** para visualizaciones de calidad de publicación (tablero, políticas y curvas de aprendizaje).
- **`reticulate`** como puente con `gymnasium` (entorno conda `rl-docencia`) en el bloque de CartPole.

La estructura sigue, por orden, los mismos bloques que el cuaderno Python:

1. **El entorno Cliff Walking**: GridWorld $4\times 12$ con precipicio (clase `R6`).
2. **La política $\varepsilon$-greedy**: el mecanismo de exploración.
3. **SARSA**: control *on-policy*.
4. **Q-Learning**: control *off-policy*.
5. **Réplica del experimento canónico** (Sutton & Barto, figura 6.4).
6. **Las políticas aprendidas**: ruta segura frente a ruta óptima.
7. **El efecto de $\varepsilon$ y su decaimiento** (condición GLIE).
8. **Primer contacto con CartPole**: discretización y antesala de la aproximación de funciones.

> **Compatibilidad.** Se ejecuta en R 4.3 o superior, tanto en entorno local (Windows/Linux/macOS) como en Google Colab con el kernel `ir`. La combinación recomendada por el docente es **R 4.5.2 + Rtools 4.4 + Miniconda** con el entorno conda `rl-docencia` (Python 3.11 con `gymnasium`). Los bloques 1 a 7 son R puro (`R6`, `ggplot2`); el bloque 8 requiere `reticulate` y `gymnasium`.

---

### Notación recurrente

| Símbolo | Significado |
|---------|-------------|
| $Q^\pi(s,a)$ | Función de valor de acción bajo la política $\pi$ |
| $q_*(s,a)$ | Función de valor de acción óptima |
| $\delta_t$ | Error de diferencia temporal en el paso $t$ |
| $\alpha$ | Paso de aprendizaje |
| $\gamma$ | Factor de descuento, $\gamma \in (0,1]$ |
| $\varepsilon$ | Probabilidad de exploración |

## Bloque 0 — Configuración del entorno

Cargamos los paquetes, fijamos la paleta institucional y el tema gráfico, y establecemos las semillas de reproducibilidad. Siguiendo la convención del curso, `SEMILLA_GLOBAL` gobierna todos los experimentos estadísticos. Introducimos además `SEMILLA_VIS`, cuya razón de ser se explica en el Bloque 5: el generador Mersenne-Twister de R difiere del PCG64 de Python, de modo que ciertas trayectorias voraces individuales deben fijarse con una semilla representativa.

In [ ]:
# --- Paso 1: paquetes R (con soporte para Google Colab) ---
ruta_setup_personal <- "/content/drive/MyDrive/R_Colab/setup_R_colab.R"
if (file.exists(ruta_setup_personal)) {
  cat("Cargando setup personal desde Drive...\n")
  source(ruta_setup_personal)
} else {
  paquetes_clase4 <- c("R6", "ggplot2", "dplyr", "tidyr")
  faltantes <- paquetes_clase4[!sapply(paquetes_clase4,
                  function(p) requireNamespace(p, quietly = TRUE))]
  if (length(faltantes) > 0) install.packages(faltantes, quiet = TRUE)
}

suppressMessages({
  library(R6); library(ggplot2); library(dplyr); library(tidyr)
})

# Semillas de reproducibilidad
SEMILLA_GLOBAL <- 42L   # experimentos estadisticos (promedios sobre replicas)
SEMILLA_VIS    <- 21L   # visualizaciones de politica de un unico episodio (vease Bloque 5)
set.seed(SEMILLA_GLOBAL)

# Paleta institucional (consistente con la presentacion y el cuaderno Python)
COLOR_NAVY         <- "#1E3A5F"
COLOR_ORANGE       <- "#D86A2C"
COLOR_TEAL         <- "#2E8A99"
COLOR_LIGHT_TEAL   <- "#9FD3D8"
COLOR_LIGHT_ORANGE <- "#F2A06B"
COLOR_GRAY         <- "#4A4A4A"
COLOR_RED          <- "#B23A48"
COLOR_GREEN        <- "#2E7D5B"

# Tema grafico institucional
tema_austral <- theme_minimal(base_size = 11) +
  theme(
    plot.title = element_text(color = COLOR_NAVY, face = "bold", size = 12),
    axis.title = element_text(color = COLOR_NAVY),
    axis.text  = element_text(color = COLOR_GRAY),
    panel.grid.minor = element_blank(),
    legend.position = "right"
  )
theme_set(tema_austral)

cat("Entorno R configurado correctamente.\n")
cat("R version:", R.version.string, "\n")

## Bloque 1 — El entorno *Cliff Walking*

Reproducimos el entorno canónico de Sutton & Barto (ejemplo 6.6): una rejilla de $4\times 12$ celdas. El agente parte de la esquina inferior izquierda ($S$) y debe alcanzar la inferior derecha ($G$). A lo largo del borde inferior se extiende **el precipicio**.

- Cada transición ordinaria cuesta $-1$.
- Pisar el precipicio acarrea $-100$ y devuelve al agente al inicio.
- El episodio concluye únicamente en la meta. El problema es **determinista** y **no descontado** ($\gamma = 1$).

A diferencia de Python (índices $0$-based), R indexa desde $1$. Codificamos cada estado como $\text{idx} = \text{fila}\cdot n_{\text{cols}} + \text{columna} + 1$, de modo que el inicio es el estado $37$ y la meta el $48$. La clase se define con `R6`, cuyo método `paso()` devuelve una lista `list(siguiente, recompensa, terminal)`.

In [ ]:
CliffWalking <- R6Class("CliffWalking",
  public = list(
    n_filas = NULL, n_cols = NULL, inicio = NULL, meta = NULL,
    precipicio_idx = NULL, acciones = NULL, nombres_acciones = NULL,
    flechas = NULL, n_estados = NULL, n_acciones = NULL,

    initialize = function(n_filas = 4L, n_cols = 12L) {
      self$n_filas <- n_filas
      self$n_cols  <- n_cols
      self$inicio  <- c(n_filas - 1L, 0L)            # coordenadas 0-based (fila, col)
      self$meta    <- c(n_filas - 1L, n_cols - 1L)
      # Acciones: 1=arriba, 2=derecha, 3=abajo, 4=izquierda
      self$acciones <- list(c(-1L, 0L), c(0L, 1L), c(1L, 0L), c(0L, -1L))
      self$nombres_acciones <- c("arriba", "derecha", "abajo", "izquierda")
      self$flechas <- c("\u2191", "\u2192", "\u2193", "\u2190")
      self$n_estados  <- n_filas * n_cols
      self$n_acciones <- length(self$acciones)
      # El precipicio ocupa la fila inferior, columnas 1..(n_cols-2)
      cols_prec <- 1:(n_cols - 2)
      self$precipicio_idx <- sapply(cols_prec, function(cc) self$idx(c(n_filas - 1L, cc)))
    },

    idx = function(estado) estado[1] * self$n_cols + estado[2] + 1L,   # 1-based

    coord = function(indice) {
      i0 <- indice - 1L
      c(i0 %/% self$n_cols, i0 %% self$n_cols)
    },

    reset = function() self$idx(self$inicio),

    paso = function(indice_estado, accion) {
      rc <- self$coord(indice_estado); fila <- rc[1]; col <- rc[2]
      d  <- self$acciones[[accion]]
      nf <- min(max(fila + d[1], 0L), self$n_filas - 1L)
      nc <- min(max(col  + d[2], 0L), self$n_cols  - 1L)
      idx_sig <- self$idx(c(nf, nc))
      if (idx_sig %in% self$precipicio_idx)
        return(list(siguiente = self$idx(self$inicio), recompensa = -100, terminal = FALSE))
      if (nf == self$meta[1] && nc == self$meta[2])
        return(list(siguiente = idx_sig, recompensa = -1, terminal = TRUE))
      list(siguiente = idx_sig, recompensa = -1, terminal = FALSE)
    }
  )
)

env <- CliffWalking$new()
cat(sprintf("Cliff Walking: %d estados, %d acciones.\n", env$n_estados, env$n_acciones))
cat(sprintf("Inicio: idx %d  |  Meta: idx %d  |  Precipicio: %d celdas (idx %d a %d).\n",
            env$reset(), env$idx(env$meta), length(env$precipicio_idx),
            min(env$precipicio_idx), max(env$precipicio_idx)))

### 1.2. Visualización del tablero

Definimos una función que construye el tablero con `ggplot2`, resaltando inicio, meta y precipicio. La reutilizaremos para superponer las políticas aprendidas. El eje vertical se invierte (`scale_y_reverse`) para que la fila $0$ quede arriba, como en la convención gráfica del cuaderno Python.

In [ ]:
# Data frame de celdas, etiquetadas por tipo
celdas_df <- function(env) {
  df <- expand.grid(fila = 0:(env$n_filas - 1), col = 0:(env$n_cols - 1))
  df$tipo <- "normal"
  for (k in seq_len(nrow(df))) {
    idx <- env$idx(c(df$fila[k], df$col[k]))
    if (idx %in% env$precipicio_idx)                                  df$tipo[k] <- "precipicio"
    else if (df$fila[k] == env$inicio[1] && df$col[k] == env$inicio[2]) df$tipo[k] <- "inicio"
    else if (df$fila[k] == env$meta[1]   && df$col[k] == env$meta[2])   df$tipo[k] <- "meta"
  }
  df
}

dibujar_tablero <- function(env, titulo = "") {
  df <- celdas_df(env)
  colores_tipo <- c(normal = "white", precipicio = COLOR_RED,
                    inicio = COLOR_NAVY, meta = COLOR_TEAL)
  ggplot(df, aes(col, fila)) +
    geom_tile(aes(fill = tipo), color = "grey70", linewidth = 0.3, alpha = 0.65) +
    scale_fill_manual(values = colores_tipo, guide = "none") +
    annotate("text", x = env$inicio[2], y = env$inicio[1], label = "S",
             fontface = "bold", color = "white") +
    annotate("text", x = env$meta[2], y = env$meta[1], label = "G",
             fontface = "bold", color = "white") +
    annotate("text", x = (env$n_cols - 1) / 2, y = env$n_filas - 1,
             label = "El precipicio (-100)", color = "white", size = 3) +
    scale_y_reverse() + coord_equal() +
    labs(title = titulo, x = NULL, y = NULL) +
    theme(axis.text = element_blank(), panel.grid = element_blank())
}

options(repr.plot.width = 9, repr.plot.height = 3.2)
print(dibujar_tablero(env, "Entorno Cliff Walking (4 x 12)"))

## Bloque 2 — La política $\varepsilon$-greedy

El control libre de modelo exige explorar: una política estrictamente voraz desde el inicio condenaría al ostracismo a las acciones óptimas penalizadas por azar. La política $\varepsilon$-greedy resuelve el compromiso:

$$
\pi(a\mid s)=
\begin{cases}
1-\varepsilon+\dfrac{\varepsilon}{|\mathcal{A}(s)|}, & a=\arg\max_{a'}Q(s,a'),\\[0.6em]
\dfrac{\varepsilon}{|\mathcal{A}(s)|}, & \text{en otro caso.}
\end{cases}
$$

El desempate entre acciones de igual valor se resuelve aleatoriamente, detalle que evita sesgos sistemáticos hacia la primera acción. Empleamos `sample.int` (no `sample`, que tiene un comportamiento peligroso ante un único entero).

In [ ]:
epsilon_greedy <- function(Q, s, n_acciones, epsilon) {
  if (runif(1) < epsilon) return(sample.int(n_acciones, 1))
  valores <- Q[s, ]
  maximos <- which(valores == max(valores))
  if (length(maximos) == 1L) maximos else maximos[sample.int(length(maximos), 1)]
}

## Bloque 3 — SARSA (control *on-policy*)

SARSA actualiza la estimación con la **acción efectivamente seleccionada** en el sucesor, $A_{t+1}\sim\pi$:

$$
Q(S_t,A_t)\leftarrow Q(S_t,A_t)+\alpha\bigl[\,R_{t+1}+\gamma\,Q(S_{t+1},A_{t+1})-Q(S_t,A_t)\,\bigr].
$$

Al evaluar la misma política que ejecuta —exploración incluida—, SARSA aprende a ser *prudente*. En el estado terminal se respeta $Q(\text{terminal},\cdot)=0$, de modo que el objetivo se reduce a la recompensa final. Sembramos el generador al inicio de cada llamada (`set.seed(semilla)`) para reproducibilidad por réplica.

In [ ]:
sarsa <- function(env, n_episodios, alpha = 0.5, gamma = 1.0, epsilon = 0.1,
                  semilla = 0, max_pasos = 1000, epsilon_fn = NULL) {
  set.seed(semilla)
  Q <- matrix(0, nrow = env$n_estados, ncol = env$n_acciones)
  retornos <- numeric(n_episodios)
  for (ep in seq_len(n_episodios)) {
    eps <- if (!is.null(epsilon_fn)) epsilon_fn(ep - 1L) else epsilon
    s <- env$reset()
    a <- epsilon_greedy(Q, s, env$n_acciones, eps)
    G <- 0; pasos <- 0L; terminado <- FALSE
    while (!terminado && pasos < max_pasos) {
      res <- env$paso(s, a); s2 <- res$siguiente; r <- res$recompensa; terminado <- res$terminal
      G <- G + r; pasos <- pasos + 1L
      if (terminado) {
        Q[s, a] <- Q[s, a] + alpha * (r - Q[s, a])
      } else {
        a2 <- epsilon_greedy(Q, s2, env$n_acciones, eps)
        Q[s, a] <- Q[s, a] + alpha * (r + gamma * Q[s2, a2] - Q[s, a])
        s <- s2; a <- a2
      }
    }
    retornos[ep] <- G
  }
  list(Q = Q, retornos = retornos)
}

# Prueba rapida
res_sarsa <- sarsa(env, n_episodios = 500, semilla = SEMILLA_GLOBAL)
cat(sprintf("SARSA - retorno medio en los ultimos 50 episodios: %.1f\n",
            mean(tail(res_sarsa$retornos, 50))))

## Bloque 4 — Q-Learning (control *off-policy*)

Una única sustitución frente a SARSA transforma el método: el sucesor $\gamma\,Q(S_{t+1},A_{t+1})$ cede su lugar al máximo sobre las acciones,

$$
Q(S_t,A_t)\leftarrow Q(S_t,A_t)+\alpha\bigl[\,R_{t+1}+\gamma\,\max_{a}Q(S_{t+1},a)-Q(S_t,A_t)\,\bigr].
$$

El objetivo deja de depender de la próxima acción ejecutada: codifica la política **voraz** mientras el agente se comporta de forma **exploratoria**. De ahí su carácter *off-policy*. Adviértase que ya no se elige $A'$.

In [ ]:
q_learning <- function(env, n_episodios, alpha = 0.5, gamma = 1.0, epsilon = 0.1,
                       semilla = 0, max_pasos = 1000, epsilon_fn = NULL) {
  set.seed(semilla)
  Q <- matrix(0, nrow = env$n_estados, ncol = env$n_acciones)
  retornos <- numeric(n_episodios)
  for (ep in seq_len(n_episodios)) {
    eps <- if (!is.null(epsilon_fn)) epsilon_fn(ep - 1L) else epsilon
    s <- env$reset(); G <- 0; pasos <- 0L; terminado <- FALSE
    while (!terminado && pasos < max_pasos) {
      a <- epsilon_greedy(Q, s, env$n_acciones, eps)
      res <- env$paso(s, a); s2 <- res$siguiente; r <- res$recompensa; terminado <- res$terminal
      G <- G + r; pasos <- pasos + 1L
      if (terminado) {
        Q[s, a] <- Q[s, a] + alpha * (r - Q[s, a])
      } else {
        Q[s, a] <- Q[s, a] + alpha * (r + gamma * max(Q[s2, ]) - Q[s, a])
      }
      s <- s2
    }
    retornos[ep] <- G
  }
  list(Q = Q, retornos = retornos)
}

# Prueba rapida
res_ql <- q_learning(env, n_episodios = 500, semilla = SEMILLA_GLOBAL)
cat(sprintf("Q-Learning - retorno medio en los ultimos 50 episodios: %.1f\n",
            mean(tail(res_ql$retornos, 50))))

## Bloque 5 — Réplica del experimento canónico (Sutton & Barto, figura 6.4)

Comparamos el **retorno por episodio** de ambos métodos durante el aprendizaje, con $\varepsilon=0.1$, $\alpha=0.5$ y $\gamma=1$, promediando sobre $N$ réplicas independientes y suavizando con media móvil. La predicción teórica: SARSA aprende la **ruta segura** y obtiene un retorno en línea *superior*; Q-Learning, que aprende la ruta óptima pero la ejecuta con ruido, se despeña con frecuencia y exhibe un retorno *inferior*.

> **Sobre la reproducibilidad y los generadores aleatorios.** Este cuaderno emplea el **Mersenne-Twister** de R (vía `set.seed`), mientras que el cuaderno Python emplea el **PCG64** de `numpy.random.default_rng`. Las trayectorias episódicas individuales no coinciden numéricamente entre ambas versiones, pero las conclusiones **estadísticas** —el orden de las curvas, los retornos asintóticos, la ruta segura frente a la óptima— son idénticas. Esta divergencia ilustra un principio importante: las garantías de convergencia son *distribucionales*, no trayectoria a trayectoria. Por la misma razón, las visualizaciones de política de un único episodio (Bloques 6 y 7) emplean `SEMILLA_VIS`, pues ciertas semillas producen políticas voraces degeneradas en celdas poco visitadas; los promedios de este bloque, en cambio, usan `SEMILLA_GLOBAL`.

In [ ]:
correr_replicas_control <- function(metodo, n_replicas, n_episodios, semilla_base, ...) {
  matriz <- matrix(0, nrow = n_replicas, ncol = n_episodios)
  for (i in seq_len(n_replicas))
    matriz[i, ] <- metodo(env, n_episodios = n_episodios,
                          semilla = semilla_base + (i - 1L), ...)$retornos
  matriz
}

media_movil <- function(x, ventana = 10) {
  cx <- cumsum(c(0, x))
  (cx[(ventana + 1):length(cx)] - cx[1:(length(cx) - ventana)]) / ventana
}

# N=100 replicas (el cuaderno Python usa 150; 100 basta para la reduccion de varianza)
N_REPLICAS  <- 100L
N_EPISODIOS <- 500L

cat(sprintf("Ejecutando %d replicas de SARSA ...\n", N_REPLICAS))
ret_sarsa <- correr_replicas_control(sarsa, N_REPLICAS, N_EPISODIOS, SEMILLA_GLOBAL,
                                     alpha = 0.5, epsilon = 0.1, gamma = 1.0)
cat(sprintf("Ejecutando %d replicas de Q-Learning ...\n", N_REPLICAS))
ret_ql <- correr_replicas_control(q_learning, N_REPLICAS, N_EPISODIOS, SEMILLA_GLOBAL,
                                  alpha = 0.5, epsilon = 0.1, gamma = 1.0)
cat("Replicas completadas.\n")

In [ ]:
VENTANA <- 10L
ejes_x  <- VENTANA:N_EPISODIOS
df_64 <- rbind(
  data.frame(Episodio = ejes_x, Retorno = media_movil(colMeans(ret_sarsa), VENTANA),
             Metodo = "SARSA (on-policy)"),
  data.frame(Episodio = ejes_x, Retorno = media_movil(colMeans(ret_ql), VENTANA),
             Metodo = "Q-Learning (off-policy)")
)

options(repr.plot.width = 10, repr.plot.height = 5.5)
p_64 <- ggplot(df_64, aes(Episodio, Retorno, color = Metodo)) +
  geom_line(linewidth = 1) +
  geom_hline(yintercept = -13, linetype = "dotted", color = COLOR_GRAY) +
  annotate("text", x = N_EPISODIOS * 0.5, y = -8, label = "ruta optima (-13)",
           color = COLOR_GRAY, size = 3) +
  scale_color_manual(values = c("SARSA (on-policy)" = COLOR_NAVY,
                                "Q-Learning (off-policy)" = COLOR_ORANGE)) +
  ylim(-100, 0) +
  labs(title = sprintf("Cliff Walking: retorno en linea (promedio de %d replicas)", N_REPLICAS),
       x = "Episodio", y = "Suma de recompensas por episodio") +
  theme(legend.position = c(0.75, 0.2))
print(p_64)

cat("Retorno medio asintotico (ultimos 100 episodios):\n")
cat(sprintf("  SARSA      : %.1f\n", mean(colMeans(ret_sarsa)[401:500])))
cat(sprintf("  Q-Learning : %.1f\n", mean(colMeans(ret_ql)[401:500])))

**Lectura del resultado.** La curva de SARSA se sitúa por encima de la de Q-Learning durante todo el aprendizaje: es la *paradoja del rendimiento en línea*. Q-Learning aprende la política óptima —la que bordea el precipicio—, pero al ejecutarla con exploración $\varepsilon$ cae al vacío con frecuencia. SARSA, consciente de su torpeza exploratoria, rodea el peligro. La línea en $-13$ marca el retorno de la ruta óptima sin error (13 pasos de coste $-1$).

## Bloque 6 — Las políticas aprendidas: ruta segura frente a ruta óptima

Extraemos la política voraz de cada matriz $Q$ y trazamos la trayectoria que el agente seguiría sin exploración. Conforme a la nota del Bloque 5, entrenamos estas matrices con `SEMILLA_VIS` para obtener trayectorias representativas.

In [ ]:
politica_voraz <- function(env, Q) apply(Q, 1, which.max)

trayectoria_voraz <- function(env, Q, max_pasos = 200) {
  s <- env$reset(); camino <- list(env$coord(s))
  for (i in seq_len(max_pasos)) {
    a   <- which.max(Q[s, ])
    res <- env$paso(s, a)
    camino[[length(camino) + 1L]] <- env$coord(res$siguiente)
    if (res$terminal) break
    if (res$siguiente == s) break   # estancamiento
    s <- res$siguiente
  }
  camino
}

dibujar_politica <- function(env, Q, titulo) {
  p <- dibujar_tablero(env, titulo)
  df <- celdas_df(env); pol <- politica_voraz(env, Q)
  df$flecha <- NA_character_
  for (k in seq_len(nrow(df)))
    if (df$tipo[k] == "normal")
      df$flecha[k] <- env$flechas[pol[env$idx(c(df$fila[k], df$col[k]))]]
  dff <- df[!is.na(df$flecha), ]
  cam <- trayectoria_voraz(env, Q)
  camdf <- data.frame(fila = sapply(cam, `[`, 1), col = sapply(cam, `[`, 2))
  p +
    geom_text(data = dff, aes(col, fila, label = flecha),
              inherit.aes = FALSE, color = COLOR_GRAY, size = 4) +
    geom_path(data = camdf, aes(col, fila), inherit.aes = FALSE,
              color = COLOR_GREEN, linewidth = 1.1, alpha = 0.85)
}

# Entrenamiento para visualizacion (semilla representativa)
Q_sarsa_vis <- sarsa(env,      n_episodios = 500, semilla = SEMILLA_VIS)$Q
Q_ql_vis    <- q_learning(env, n_episodios = 500, semilla = SEMILLA_VIS)$Q

options(repr.plot.width = 9, repr.plot.height = 3.2)
print(dibujar_politica(env, Q_sarsa_vis, "Politica voraz de SARSA - la ruta segura"))
print(dibujar_politica(env, Q_ql_vis,    "Politica voraz de Q-Learning - la ruta optima"))

cat("Longitud de la ruta voraz final (optimo = 13 pasos):\n")
cat(sprintf("  SARSA      : %d\n", length(trayectoria_voraz(env, Q_sarsa_vis)) - 1))
cat(sprintf("  Q-Learning : %d\n", length(trayectoria_voraz(env, Q_ql_vis)) - 1))

La evidencia visual confirma la teoría: **SARSA** se aleja del borde, recorriendo la fila superior antes de descender; **Q-Learning** se pega al precipicio, la trayectoria de máximo retorno cuando se la sigue sin error. Ambas políticas son racionales bajo su propio criterio de evaluación.

## Bloque 7 — El efecto de $\varepsilon$ y su decaimiento (condición GLIE)

Una exploración constante impide alcanzar una política óptima determinista. La condición **GLIE** prescribe decaer $\varepsilon$ hacia cero conservando visitas infinitas a cada par. Empleamos un decaimiento armónico $\varepsilon_k = \varepsilon_0 / (1 + k\cdot\tau)$ y examinamos dos efectos: la **política voraz** finalmente aprendida y el **retorno en línea** durante el aprendizaje.

In [ ]:
epsilon_decreciente <- function(eps0 = 0.5, tau = 0.01) function(k) eps0 / (1.0 + k * tau)

N_EP_DECAY <- 1000L
Q_sarsa_d <- sarsa(env,      n_episodios = N_EP_DECAY, semilla = SEMILLA_VIS,
                   epsilon_fn = epsilon_decreciente(0.5, 0.01))$Q
Q_ql_d    <- q_learning(env, n_episodios = N_EP_DECAY, semilla = SEMILLA_VIS,
                        epsilon_fn = epsilon_decreciente(0.5, 0.01))$Q

options(repr.plot.width = 9, repr.plot.height = 3.2)
print(dibujar_politica(env, Q_sarsa_d, "SARSA con epsilon decreciente - conserva la ruta prudente"))
print(dibujar_politica(env, Q_ql_d,    "Q-Learning con epsilon decreciente - la ruta optima"))

cat("Longitud de la ruta voraz final (optimo = 13 pasos):\n")
cat(sprintf("  SARSA      (eps decreciente): %d\n", length(trayectoria_voraz(env, Q_sarsa_d)) - 1))
cat(sprintf("  Q-Learning (eps decreciente): %d\n", length(trayectoria_voraz(env, Q_ql_d)) - 1))

**Una sutileza que conviene no ocultar.** La política voraz de Q-Learning coincide ya con la ruta óptima, mientras que la de SARSA preserva un margen de seguridad. La razón es doble. Primero, la convergencia de SARSA a $q_*$ es *asintótica* y exige las **dos** condiciones del teorema (Singh y col., 2000): no basta con que $\varepsilon$ decaiga (GLIE); también el paso $\alpha$ debe satisfacer Robbins-Monro. Con $\alpha=0.5$ constante, la estimación nunca se asienta del todo y la política on-policy retiene su cautela. Segundo, las celdas contiguas al precipicio se visitan cada vez menos al desvanecerse la exploración, conservando los valores pesimistas heredados de las caídas tempranas. La distinción on-policy / off-policy se revela así como una **velocidad y un sesgo de aprendizaje** distintos, no como un destino último diferente.

In [ ]:
# El beneficio concreto del decaimiento: el retorno EN LINEA de Q-Learning
N_REP_COMP <- 30L
ret_ql_const  <- correr_replicas_control(q_learning, N_REP_COMP, N_EP_DECAY, SEMILLA_GLOBAL,
                                         alpha = 0.5, epsilon = 0.1, gamma = 1.0)
ret_ql_decay  <- correr_replicas_control(q_learning, N_REP_COMP, N_EP_DECAY, SEMILLA_GLOBAL,
                                         alpha = 0.5, gamma = 1.0,
                                         epsilon_fn = epsilon_decreciente(0.5, 0.01))
ret_sarsa_cst <- correr_replicas_control(sarsa, N_REP_COMP, N_EP_DECAY, SEMILLA_GLOBAL,
                                         alpha = 0.5, epsilon = 0.1, gamma = 1.0)

V2 <- 20L; ex2 <- V2:N_EP_DECAY
df_on <- rbind(
  data.frame(Episodio = ex2, Retorno = media_movil(colMeans(ret_sarsa_cst), V2),
             Serie = "SARSA (eps=0.1 constante)"),
  data.frame(Episodio = ex2, Retorno = media_movil(colMeans(ret_ql_const), V2),
             Serie = "Q-Learning (eps=0.1 constante)"),
  data.frame(Episodio = ex2, Retorno = media_movil(colMeans(ret_ql_decay), V2),
             Serie = "Q-Learning (eps decreciente)")
)

options(repr.plot.width = 10, repr.plot.height = 5.5)
p_on <- ggplot(df_on, aes(Episodio, Retorno, color = Serie)) +
  geom_line(linewidth = 1) +
  geom_hline(yintercept = -13, linetype = "dotted", color = COLOR_GRAY) +
  scale_color_manual(values = c("SARSA (eps=0.1 constante)" = COLOR_NAVY,
                                "Q-Learning (eps=0.1 constante)" = COLOR_ORANGE,
                                "Q-Learning (eps decreciente)" = COLOR_GREEN)) +
  ylim(-100, 0) +
  labs(title = "El decaimiento de epsilon rescata el retorno en linea de Q-Learning",
       x = "Episodio", y = "Suma de recompensas por episodio") +
  theme(legend.position = c(0.72, 0.2))
print(p_on)

cat("Retorno medio en linea (ultimos 100 episodios):\n")
cat(sprintf("  SARSA      (eps=0.1)        : %.1f\n", mean(colMeans(ret_sarsa_cst)[(N_EP_DECAY-99):N_EP_DECAY])))
cat(sprintf("  Q-Learning (eps=0.1)        : %.1f\n", mean(colMeans(ret_ql_const)[(N_EP_DECAY-99):N_EP_DECAY])))
cat(sprintf("  Q-Learning (eps decreciente): %.1f\n", mean(colMeans(ret_ql_decay)[(N_EP_DECAY-99):N_EP_DECAY])))

Al desvanecerse $\varepsilon$, Q-Learning deja de despeñarse por sus propios pasos exploratorios: su retorno en línea remonta hacia el óptimo de $-13$, alcanzando a SARSA hacia el final. La lección operativa: si importa el desempeño *durante* el aprendizaje en un entorno peligroso, conviene decaer la exploración o preferir un método on-policy.

## Bloque 8 — Primer contacto con CartPole: la antesala de la aproximación de funciones

El formato tabular reposa sobre un número finito y manejable de estados. CartPole (Gymnasium) lo quiebra, pues su estado es **continuo** —posición y velocidad del carro, ángulo y velocidad angular del péndulo—. Para aplicar Q-Learning tabular **discretizamos** cada dimensión, lo que multiplica la cardinalidad del espacio: la *maldición de la dimensionalidad* en acción, y la motivación directa de la aproximación de funciones de la Clase 5.

Empleamos **`reticulate`** para acceder a `gymnasium` desde R, a través del entorno conda `rl-docencia`. La discretización en R usa `findInterval`, equivalente exacto de `numpy.digitize` para fronteras crecientes.

In [ ]:
# Puente con Python via reticulate (entorno conda 'rl-docencia')
Sys.setenv(PYTHONNOUSERSITE = "1")
suppressMessages(library(reticulate))

ok_gym <- tryCatch({
  use_condaenv("rl-docencia", required = TRUE)
  gym <<- import("gymnasium")
  TRUE
}, error = function(e) {
  message("No se pudo activar el entorno conda 'rl-docencia': ", conditionMessage(e))
  message("Intentando importar gymnasium del Python activo...")
  tryCatch({ gym <<- import("gymnasium"); TRUE },
           error = function(e2) {
             message("gymnasium no disponible. En Google Colab, ejecute en una celda:")
             message("    reticulate::py_install('gymnasium', pip = TRUE)")
             FALSE
           })
})
if (ok_gym) cat("gymnasium", gym$`__version__`, "\n")

In [ ]:
# Discretizacion del estado continuo de CartPole
LIMITES <- list(c(-2.4, 2.4), c(-3.0, 3.0), c(-0.21, 0.21), c(-3.5, 3.5))
N_BINS  <- c(3L, 3L, 8L, 8L)   # mas finura en angulo y velocidad angular

bordes <- lapply(seq_along(LIMITES), function(i)
  seq(LIMITES[[i]][1], LIMITES[[i]][2], length.out = N_BINS[i] + 1)[2:N_BINS[i]])
N_ESTADOS_CP <- prod(N_BINS)

discretizar <- function(obs) {
  idx <- 0L
  for (i in seq_along(obs)) idx <- idx * N_BINS[i] + findInterval(obs[i], bordes[[i]])
  idx + 1L   # indice 1-based para R
}

cat(sprintf("Espacio discretizado de CartPole: %d estados (= %s).\n",
            N_ESTADOS_CP, paste(N_BINS, collapse = "x")))
cat("Comparese con los 48 estados de Cliff Walking: la explosion combinatoria es evidente.\n")

In [ ]:
q_learning_cartpole <- function(n_episodios = 600, alpha = 0.1, gamma = 0.99,
                                eps0 = 1.0, eps_min = 0.02, decay = 0.99,
                                semilla = SEMILLA_GLOBAL) {
  entorno <- gym$make("CartPole-v1")
  n_acc   <- as.integer(entorno$action_space$n)
  set.seed(semilla)
  Q <- matrix(0, nrow = N_ESTADOS_CP, ncol = n_acc)
  retornos <- numeric(n_episodios)
  eps <- eps0
  for (ep in seq_len(n_episodios)) {
    rr  <- entorno$reset(seed = as.integer(semilla + ep)); obs <- rr[[1]]
    s   <- discretizar(obs)
    fin <- FALSE; G <- 0
    while (!fin) {
      if (runif(1) < eps) a <- sample.int(n_acc, 1) else a <- which.max(Q[s, ])
      res <- entorno$step(as.integer(a - 1L))   # accion 0-based para gym
      obs2 <- res[[1]]; r <- res[[2]]; terminado <- res[[3]]; truncado <- res[[4]]
      s2  <- discretizar(obs2)
      objetivo <- r + if (terminado) 0 else gamma * max(Q[s2, ])
      Q[s, a] <- Q[s, a] + alpha * (objetivo - Q[s, a])
      s <- s2; G <- G + r
      fin <- terminado || truncado
    }
    retornos[ep] <- G
    eps <- max(eps_min, eps * decay)
  }
  entorno$close()
  list(Q = Q, retornos = retornos)
}

if (ok_gym) {
  cat("Entrenando Q-Learning tabular sobre CartPole discretizado ...\n")
  # n_episodios moderado por el coste de reticulate; el cuaderno Python usa 2000.
  res_cp <- q_learning_cartpole(n_episodios = 600)
  ret_cp <- res_cp$retornos
  cat(sprintf("Retorno medio (ultimos 100 episodios): %.1f  (maximo posible: 500).\n",
              mean(tail(ret_cp, 100))))
} else {
  cat("Bloque 8 omitido: gymnasium no disponible en este entorno.\n")
}

In [ ]:
if (ok_gym) {
  options(repr.plot.width = 10, repr.plot.height = 5)
  df_cp <- data.frame(Episodio = seq_along(ret_cp), Retorno = ret_cp)
  Vcp <- 20L
  df_mm <- data.frame(Episodio = Vcp:length(ret_cp), Retorno = media_movil(ret_cp, Vcp))
  p_cp <- ggplot() +
    geom_line(data = df_cp, aes(Episodio, Retorno), color = COLOR_LIGHT_TEAL, linewidth = 0.4) +
    geom_line(data = df_mm, aes(Episodio, Retorno), color = COLOR_NAVY, linewidth = 1) +
    geom_hline(yintercept = 500, linetype = "dotted", color = COLOR_GRAY) +
    labs(title = "Q-Learning tabular sobre CartPole discretizado",
         x = "Episodio", y = "Retorno (pasos de equilibrio)")
  print(p_cp)
}

El agente tabular aprende a sostener el péndulo durante un número creciente de pasos, pero la discretización paga un precio: ignora la vecindad entre estados parecidos y multiplica la memoria requerida. En la **Clase 5** sustituiremos la tabla por una función parametrizada $\hat{q}(s,a;\mathbf{w})$ —primero lineal, luego una red neuronal—, lo que dará lugar a DQN. Q-Learning, el algoritmo de hoy, será su corazón conceptual.

## Ejercicios propuestos

Los siguientes ejercicios extienden las implementaciones de este cuaderno. El componente evaluado de la Unidad 3 tiene como plazo de entrega el **1 de junio, 23:59 hs**.

### Ejercicios obligatorios (formales)

1. **Ejercicio teórico 1.** Demuestre el teorema de mejora para políticas $\varepsilon$-greedy: si $\pi'$ es la política $\varepsilon$-greedy respecto de $Q^\pi$, entonces $Q^\pi(s,\pi'(s))\ge V^\pi(s)$ para todo $s$, y por ende $V^{\pi'}\ge V^\pi$. Identifique el papel de la cota $\varepsilon/|\mathcal{A}(s)|$.

2. **Ejercicio teórico 2.** Exhiba la sustitución exacta de la política $\pi$ que reduce la actualización de *Expected SARSA*,
   $$Q(S_t,A_t)\leftarrow Q(S_t,A_t)+\alpha\Bigl[R_{t+1}+\gamma\textstyle\sum_a \pi(a\mid S_{t+1})Q(S_{t+1},a)-Q(S_t,A_t)\Bigr],$$
   a la actualización de Q-Learning. Discuta por qué ello revela su carácter *off-policy*.

3. **Ejercicio computacional 1.** Reproduzca la figura del Bloque 5 con $N=500$ réplicas y barriendo $\varepsilon\in\{0.05, 0.1, 0.2\}$. ¿Cómo se modifica la brecha entre SARSA y Q-Learning al aumentar $\varepsilon$? Justifique.

4. **Ejercicio computacional 2.** Implemente **Expected SARSA** sobre Cliff Walking (en R) y compárelo con SARSA y Q-Learning en retorno en línea y varianza entre réplicas. ¿Confirma la reducción de varianza que predice la teoría?

### Ejercicios opcionales (de profundización)

5. **Ejercicio integrador (sesgo de maximización).** Construya el MDP de dos estados del ejemplo 6.7 de Sutton & Barto y cuantifique empíricamente la sobreestimación de Q-Learning. Implemente **doble Q-Learning** y verifique la corrección del sesgo.

6. **Ejercicio avanzado (CartPole).** Estudie el efecto de la granularidad de la discretización (`N_BINS`) sobre el desempeño y la velocidad de aprendizaje. Documente el compromiso entre resolución y número de estados, y argumente por qué la aproximación de funciones es la salida natural.

---

### Referencias del cuaderno

* Sutton, R. S., & Barto, A. G. (2018). *Reinforcement Learning: An Introduction* (2.ª ed.). MIT Press. Capítulo 6, secciones 6.4 a 6.7.
* Zhao, S. (2024). *Mathematical Foundations of Reinforcement Learning*. Springer. Capítulo 7, secciones 7.2 a 7.4.
* Watkins, C. J. C. H., & Dayan, P. (1992). Q-learning. *Machine Learning*, 8(3), 279–292.
* van Hasselt, H. (2010). Double Q-learning. *Advances in Neural Information Processing Systems*, 23.

---

*Cuaderno R preparado para la Clase 4 del curso «Aprendizaje por Refuerzo: Fundamentos y Aplicaciones», Universidad Austral, mayo de 2026. Contraparte de `Clase04_SARSA_QLearning.ipynb`.*